In [ ]:
# Example 1: Public API surface — verify top-level exports exist.
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings

assert ChatMistralAI is not None
assert MistralAIEmbeddings is not None

print(f"ChatMistralAI: {ChatMistralAI}")
print(f"MistralAIEmbeddings: {MistralAIEmbeddings}")
print(f"ChatMistralAI MRO depth: {len(ChatMistralAI.__mro__)}")
print(f"MistralAIEmbeddings MRO depth: {len(MistralAIEmbeddings.__mro__)}")


In [ ]:
# Example 2: ChatMistralAI construction with a dummy key — no network call.
from langchain_mistralai import ChatMistralAI
from pydantic import SecretStr

chat = ChatMistralAI(
    model="mistral-small-latest",
    mistral_api_key=SecretStr("sk-test-not-a-real-key"),
    temperature=0.0,
    max_tokens=64,
)

assert chat.model == "mistral-small-latest"
assert chat.temperature == 0.0
assert chat.max_tokens == 64
print(f"ChatMistralAI constructed: model={chat.model}, temperature={chat.temperature}")
print(f"_llm_type: {chat._llm_type}")


In [ ]:
# Example 3: MistralAIEmbeddings construction + tokenizer access (no network).
from langchain_mistralai import MistralAIEmbeddings
from pydantic import SecretStr

emb = MistralAIEmbeddings(
    model="mistral-embed",
    mistral_api_key=SecretStr("sk-test-not-a-real-key"),
)

assert emb.model == "mistral-embed"
# Tokenizer is lazily loaded; just verify the attribute exists and class is correct.
print(f"MistralAIEmbeddings constructed: model={emb.model}")
print(f"endpoint: {emb.endpoint}")
print(f"max_retries: {emb.max_retries}")


In [ ]:
# Example 4: Message-formatting via ChatPromptTemplate against ChatMistralAI input schema.
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{question}"),
])
messages = prompt.format_messages(question="What is 2 + 2?")

assert len(messages) == 2
assert isinstance(messages[0], SystemMessage)
assert isinstance(messages[1], HumanMessage)
assert "2 + 2" in messages[1].content
print(f"Formatted {len(messages)} messages:")
for m in messages:
    print(f"  {type(m).__name__}: {m.content!r}")


In [ ]:
# Example 5: Tool-binding API — bind a Pydantic schema as a structured tool.
from langchain_mistralai import ChatMistralAI
from pydantic import BaseModel, Field, SecretStr

class GetWeather(BaseModel):
    """Get the current weather in a given location."""
    location: str = Field(..., description="City name")
    unit: str = Field(default="celsius", description="Temperature unit")

chat = ChatMistralAI(
    model="mistral-large-latest",
    mistral_api_key=SecretStr("sk-test-not-a-real-key"),
)
chat_with_tools = chat.bind_tools([GetWeather])

assert chat_with_tools is not None
assert chat_with_tools is not chat  # bind_tools returns a new runnable
print(f"Bound tools onto ChatMistralAI; result type: {type(chat_with_tools).__name__}")
print("langchain_mistralai test notebook PASSED")
